In [ ]:
%%bash

docker compose -f /home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docker-compose.yml down minio

In [ ]:
%%bash

docker compose -f /home/brijeshdhaker/IdeaProjects/bd-notebooks-module/docker-compose.yml up -d minio


In [1]:
import os
from pyspark import SparkConf
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta import *

# Adding AWS S3 Minio configs
sparkConf = (
    SparkConf()
    .set("spark.jars.ivy","/home/brijeshdhaker/.ivy2")
    .set("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .set("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .set("spark.jars.packages","org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.797,io.delta:delta-spark_2.12:3.3.2")
    .set("spark.executor.heartbeatInterval", "300000")
    .set("spark.network.timeout", "400000")
    .set("spark.hadoop.fs.defaultFS", "s3a://defaultfs/")
    .set("spark.hadoop.fs.s3a.endpoint", "http://minio.sandbox.net:9010")
    .set("spark.hadoop.fs.s3a.access.key", "pgm2H2bR7a5kMc5XCYdO")
    .set("spark.hadoop.fs.s3a.secret.key", "zjd8T0hXFGtfemVQ6AH3yBAPASJNXNbVSx5iddqG")
    .set("spark.hadoop.fs.s3a.path.style.access", "true")
    .set("spark.hadoop.fs.s3a.aws.credentials.provider", "org.apache.hadoop.fs.s3a.SimpleAWSCredentialsProvider")
    .set("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .set("spark.hadoop.delta.enableFastS3AListFrom", "true")
    #.set("spark.eventLog.enabled", "true")
    #.set("spark.eventLog.dir", "file:///apps/var/logs/spark-events")
)

# configure the SparkSession with the configure_spark_with_delta_pip() utility function in Delta Lake:
builder = SparkSession.builder.appName("spark-deltalake").master("local[*]").config(conf=sparkConf)
spark = configure_spark_with_delta_pip(builder, extra_packages=["org.apache.hadoop:hadoop-aws:3.3.4"]).getOrCreate()

#
spark.sparkContext.setLogLevel('ERROR')
spark

#
# 
#
def display(df):
    df.show(truncate=False)

:: loading settings :: url = jar:file:/opt/conda/lib/python3.11/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/brijeshdhaker/.ivy2/cache
The jars for the packages stored in: /home/brijeshdhaker/.ivy2/jars
io.delta#delta-spark_2.12 added as a dependency
org.apache.hadoop#hadoop-aws added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-d98286d8-1ccb-4bf4-8d0b-db26a92d6e1b;1.0
	confs: [default]
	found io.delta#delta-spark_2.12;3.3.2 in local-m2-cache
	found io.delta#delta-storage;3.3.2 in local-m2-cache
	found org.antlr#antlr4-runtime;4.9.3 in local-m2-cache
	found org.apache.hadoop#hadoop-aws;3.3.4 in local-m2-cache
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in local-m2-cache
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in local-m2-cache
:: resolution report :: resolve 187ms :: artifacts dl 15ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from local-m2-cache in [default]
	io.delta#delta-spark_2.12;3.3.2 from local-m2-cache in [default]
	io.delta#delta-storage;3.3.2 from local-m2-cache in [defaul

#### Delete Existing Delta Table

In [2]:
%%bash

## Delete Existing Delta Table
aws --endpoint-url http://minio.sandbox.net:9010 s3 rm s3://defaultfs/deltalake/peoples --recursive


delete: s3://defaultfs/deltalake/peoples/_delta_log/00000000000000000000.crc
delete: s3://defaultfs/deltalake/peoples/_delta_log/00000000000000000000.json
delete: s3://defaultfs/deltalake/peoples/_delta_log/_commits/
delete: s3://defaultfs/deltalake/peoples/part-00000-1259b543-cb3e-43e8-b949-56e149e5034a-c000.snappy.parquet


#### Create Deltatable

In [18]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, TimestampType

schema = StructType([
  StructField("id", IntegerType(), True),
  StructField("firstName", StringType(), True),
  StructField("middleName", StringType(), True),
  StructField("lastName", StringType(), True),
  StructField("gender", StringType(), True),
  StructField("birthDate", TimestampType(), True),
  StructField("ssn", StringType(), True),
  StructField("salary", IntegerType(), True)
])

df = spark.read.format("csv").option("header", True).schema(schema).load("s3a://datasets/people-data/peoples.csv")
df.printSchema()

#
display(df)

root
 |-- id: integer (nullable = true)
 |-- firstName: string (nullable = true)
 |-- middleName: string (nullable = true)
 |-- lastName: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- birthDate: timestamp (nullable = true)
 |-- ssn: string (nullable = true)
 |-- salary: integer (nullable = true)

+---+----------+----------+-------------+------+-------------------+-----------+------+
|id |firstName |middleName|lastName     |gender|birthDate          |ssn        |salary|
+---+----------+----------+-------------+------+-------------------+-----------+------+
|1  |Pennie    |Carry     |Hirschmann   |F     |1955-07-02 09:30:00|981-43-9345|56172 |
|2  |An        |Amira     |Cowper       |F     |1992-02-08 10:30:00|978-97-8086|40203 |
|3  |Quyen     |Marlen    |Dome         |F     |1970-10-11 09:30:00|957-57-8246|53417 |
|4  |Coralie   |Antonina  |Marshal      |F     |1990-04-11 09:30:00|963-39-4885|94727 |
|5  |Terrie    |Wava      |Bonar        |F     |1980-01-16 10:30

#### Create Delta table

In [19]:
# Save as delta table into Minio S3
df.write.format('delta').save('/deltalake/spark_df_peoples')

# Create the table if it does not exist. Otherwise, replace the existing table.
#df.writeTo("spark_catalog.default.peoples").createOrReplace()

# If you know the table does not already exist, you can call this instead:
#df.write.saveAsTable("spark_catalog.default.peoples")

#### Read a Deltatable

In [22]:
spark_df_peoples = spark.read.format('delta').load("/deltalake/spark_df_peoples")
display(spark_df_peoples)

+---+----------+----------+-------------+------+-------------------+-----------+------+
|id |firstName |middleName|lastName     |gender|birthDate          |ssn        |salary|
+---+----------+----------+-------------+------+-------------------+-----------+------+
|1  |Pennie    |Carry     |Hirschmann   |F     |1955-07-02 09:30:00|981-43-9345|56172 |
|2  |An        |Amira     |Cowper       |F     |1992-02-08 10:30:00|978-97-8086|40203 |
|3  |Quyen     |Marlen    |Dome         |F     |1970-10-11 09:30:00|957-57-8246|53417 |
|4  |Coralie   |Antonina  |Marshal      |F     |1990-04-11 09:30:00|963-39-4885|94727 |
|5  |Terrie    |Wava      |Bonar        |F     |1980-01-16 10:30:00|964-49-8051|79908 |
|6  |Chassidy  |Concepcion|Bourthouloume|F     |1990-11-24 10:30:00|954-59-9172|64652 |
|7  |Geri      |Tambra    |Mosby        |F     |1970-12-19 10:30:00|968-16-4020|38195 |
|8  |Patria    |Nancy     |Arstall      |F     |1985-01-02 10:30:00|984-76-3770|102053|
|9  |Terese    |Alfredia  |Tocqu

In [23]:
%%sql

DESCRIBE HISTORY delta.`s3a://defaultfs/deltalake/spark_df_peoples`

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7,Field 8,Field 9,Field 10,Field 11,Field 12,Field 13,Field 14,Field 15
0,2026-03-31 12:19:20,None,None,WRITE,"{'mode': 'ErrorIfExists', 'partitionBy': '[]'}",None,None,None,None,Serializable,True,"{'numOutputRows': '1000', 'numOutputBytes': '47305', 'numFiles': '1'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2


In [29]:
from delta.tables import *

deltaTable = DeltaTable.forPath(spark, 's3a://defaultfs/deltalake/spark_df_peoples')
display(deltaTable.history())

+-------+-------------------+------+--------+---------+------------------------------------------+----+--------+---------+-----------+--------------+-------------+---------------------------------------------------------------+------------+-----------------------------------+
|version|timestamp          |userId|userName|operation|operationParameters                       |job |notebook|clusterId|readVersion|isolationLevel|isBlindAppend|operationMetrics                                               |userMetadata|engineInfo                         |
+-------+-------------------+------+--------+---------+------------------------------------------+----+--------+---------+-----------+--------------+-------------+---------------------------------------------------------------+------------+-----------------------------------+
|1      |2026-03-31 12:29:42|NULL  |NULL    |WRITE    |{mode -> Append, partitionBy -> []}       |NULL|NULL    |NULL     |0          |Serializable  |true         |{numFi

In [25]:
%%sql

INSERT INTO  delta.`s3a://defaultfs/deltalake/spark_df_peoples`
SELECT * FROM PARQUET.`s3a://defaultfs/deltalake/spark_df_peoples`;

Running query in 'SparkSession'

++
||
++
++

In [28]:
%%sql

SELECT MIN(salary), MAX(salary), COUNT(id)
FROM delta.`s3a://defaultfs/deltalake/spark_df_peoples`

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2,Field 3
-6523,134393,2000


In [31]:
%%sql 

UPDATE delta.`s3a://defaultfs/deltalake/spark_df_peoples`
SET salary = 6523
WHERE salary = -6523;

Running query in 'SparkSession'

1 rows affected.

Field 1
2


In [32]:
%%sql

DESCRIBE HISTORY delta.`s3a://defaultfs/deltalake/spark_df_peoples`

Running query in 'SparkSession'

3 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7,Field 8,Field 9,Field 10,Field 11,Field 12,Field 13,Field 14,Field 15
2,2026-03-31 12:35:33,None,None,UPDATE,"{'predicate': '[""(salary#6177 = -6523)""]'}",None,None,None,1,Serializable,False,"{'numDeletionVectorsUpdated': '0', 'numAddedFiles': '2', 'executionTimeMs': '693', 'numDeletionVectorsRemoved': '0', 'numUpdatedRows': '2', 'numRemovedFiles': '2', 'rewriteTimeMs': '200', 'numRemovedBytes': '94610', 'scanTimeMs': '490', 'numCopiedRows': '1998', 'numDeletionVectorsAdded': '0', 'numAddedChangeFiles': '0', 'numAddedBytes': '94610'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2
1,2026-03-31 12:29:42,None,None,WRITE,"{'mode': 'Append', 'partitionBy': '[]'}",None,None,None,0,Serializable,True,"{'numOutputRows': '1000', 'numOutputBytes': '47305', 'numFiles': '1'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2
0,2026-03-31 12:19:20,None,None,WRITE,"{'mode': 'ErrorIfExists', 'partitionBy': '[]'}",None,None,None,None,Serializable,True,"{'numOutputRows': '1000', 'numOutputBytes': '47305', 'numFiles': '1'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2


In [33]:
%%sql 

SELECT *
FROM delta.`s3a://defaultfs/deltalake/spark_df_peoples`
WHERE id = 1

Running query in 'SparkSession'

2 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7,Field 8
1,Pennie,Carry,Hirschmann,F,1955-07-02 09:30:00,981-43-9345,56172
1,Pennie,Carry,Hirschmann,F,1955-07-02 09:30:00,981-43-9345,56172


In [34]:
%%sql 

DELETE
FROM delta.`s3a://defaultfs/deltalake/spark_df_peoples`
WHERE id = 1

Running query in 'SparkSession'

1 rows affected.

Field 1
2


In [35]:
%%sql 

SELECT *
FROM delta.`s3a://defaultfs/deltalake/spark_df_peoples`
WHERE id = 1

Running query in 'SparkSession'

++
||
++
++

In [36]:
%%sql

DESCRIBE HISTORY delta.`s3a://defaultfs/deltalake/spark_df_peoples`

Running query in 'SparkSession'

4 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7,Field 8,Field 9,Field 10,Field 11,Field 12,Field 13,Field 14,Field 15
3,2026-03-31 12:39:30,None,None,DELETE,"{'predicate': '[""(id#7636 = 1)""]'}",None,None,None,2,Serializable,False,"{'numDeletionVectorsUpdated': '0', 'numAddedFiles': '2', 'executionTimeMs': '694', 'numDeletionVectorsRemoved': '0', 'numRemovedFiles': '2', 'rewriteTimeMs': '181', 'numRemovedBytes': '94610', 'scanTimeMs': '512', 'numCopiedRows': '1998', 'numDeletionVectorsAdded': '0', 'numAddedChangeFiles': '0', 'numDeletedRows': '2', 'numAddedBytes': '94532'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2
2,2026-03-31 12:35:33,None,None,UPDATE,"{'predicate': '[""(salary#6177 = -6523)""]'}",None,None,None,1,Serializable,False,"{'numDeletionVectorsUpdated': '0', 'numAddedFiles': '2', 'executionTimeMs': '693', 'numDeletionVectorsRemoved': '0', 'numUpdatedRows': '2', 'numRemovedFiles': '2', 'rewriteTimeMs': '200', 'numRemovedBytes': '94610', 'scanTimeMs': '490', 'numCopiedRows': '1998', 'numDeletionVectorsAdded': '0', 'numAddedChangeFiles': '0', 'numAddedBytes': '94610'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2
1,2026-03-31 12:29:42,None,None,WRITE,"{'mode': 'Append', 'partitionBy': '[]'}",None,None,None,0,Serializable,True,"{'numOutputRows': '1000', 'numOutputBytes': '47305', 'numFiles': '1'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2
0,2026-03-31 12:19:20,None,None,WRITE,"{'mode': 'ErrorIfExists', 'partitionBy': '[]'}",None,None,None,None,Serializable,True,"{'numOutputRows': '1000', 'numOutputBytes': '47305', 'numFiles': '1'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2


In [ ]:
for i in range(5):
    spark.sql(
        """
        INSERT INTO delta.`s3a://defaultfs/deltalake/spark_df_peoples`
        SELECT * FROM PARQUET.`s3a://defaultfs/deltalake/spark_df_peoples`;
        """
    )
    print(f"{i}th INSERT completed ")

0th INSERT completed 
1th INSERT completed 
2th INSERT completed 
3th INSERT completed 
4th INSERT completed 
5th INSERT completed 
6th INSERT completed 


7th INSERT completed 


8th INSERT completed 


9th INSERT completed 


10th INSERT completed 


11th INSERT completed 


12th INSERT completed 


13th INSERT completed 


ERROR:root:KeyboardInterrupt while sending command.                 (6 + 3) / 9]
Traceback (most recent call last):
  File "/opt/conda/lib/python3.11/site-packages/py4j/java_gateway.py", line 1038, in send_command
    response = connection.send_command(command)
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/site-packages/py4j/clientserver.py", line 511, in send_command
    answer = smart_decode(self.stream.readline()[:-1])
                          ^^^^^^^^^^^^^^^^^^^^^^
  File "/opt/conda/lib/python3.11/socket.py", line 718, in readinto
    return self._sock.recv_into(b)
           ^^^^^^^^^^^^^^^^^^^^^^^
KeyboardInterrupt


KeyboardInterrupt: 

In [40]:
%%sql

DESCRIBE HISTORY delta.`s3a://defaultfs/deltalake/spark_df_peoples`

Running query in 'SparkSession'

19 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7,Field 8,Field 9,Field 10,Field 11,Field 12,Field 13,Field 14,Field 15
18,2026-03-31 12:45:46,None,None,WRITE,"{'mode': 'Append', 'partitionBy': '[]'}",None,None,None,17,Serializable,True,"{'numOutputRows': '98271232', 'numOutputBytes': '247214725', 'numFiles': '9'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2
17,2026-03-31 12:45:05,None,None,WRITE,"{'mode': 'Append', 'partitionBy': '[]'}",None,None,None,16,Serializable,True,"{'numOutputRows': '49135616', 'numOutputBytes': '123995469', 'numFiles': '9'}",None,Apache-Spark/3.5.3 Delta-Lake/3.3.2


#### Python Create Deltatable

In [3]:
spark.sql("""

CREATE OR REPLACE TABLE delta.`s3a://defaultfs/deltalake/people10m` (
  id INT,
  firstName STRING,
  middleName STRING,
  lastName STRING,
  gender STRING,
  birthDate TIMESTAMP,
  ssn STRING,
  salary INT
) USING DELTA
          
""")

DataFrame[]

In [4]:
from delta import *

DeltaTable.createIfNotExists(spark).location("s3a://defaultfs/deltalake/people00m") \
    .tableName("people00m") \
    .addColumn("id", "INT") \
    .addColumn("firstName", "STRING") \
    .addColumn("middleName", "STRING") \
    .addColumn("lastName", "STRING", comment = "surname") \
    .addColumn("gender", "STRING") \
    .addColumn("birthDate", "TIMESTAMP") \
    .addColumn("ssn", "STRING") \
    .addColumn("salary", "INT") \
    .execute()

In [9]:
from delta import *
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

deltaTable = DeltaTable.forPath(spark, 's3a://defaultfs/deltalake/people00m')
deltaTable.toDF().show()


+---+---------+----------+--------+------+---------+---+------+
| id|firstName|middleName|lastName|gender|birthDate|ssn|salary|
+---+---------+----------+--------+------+---------+---+------+
+---+---------+----------+--------+------+---------+---+------+



In [17]:
%%sql

DESCRIBE HISTORY delta.`s3a://defaultfs/deltalake/people00m`

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7,Field 8,Field 9,Field 10,Field 11,Field 12,Field 13,Field 14,Field 15
0,2026-03-31 11:42:41,None,None,CREATE TABLE,"{'partitionBy': '[]', 'description': None, 'properties': '{}', 'clusterBy': '[]', 'isManaged': 'false'}",None,None,None,None,Serializable,True,{},None,Apache-Spark/3.5.3 Delta-Lake/3.3.2


In [ ]:
%%sql 

DESCRIBE HISTORY default.people12m;

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2,Field 3,Field 4,Field 5,Field 6,Field 7,Field 8,Field 9,Field 10,Field 11,Field 12,Field 13,Field 14,Field 15
0,2026-03-31 11:14:35,None,None,CREATE TABLE,"{'partitionBy': '[]', 'description': None, 'properties': '{}', 'clusterBy': '[]', 'isManaged': 'false'}",None,None,None,None,Serializable,True,{},None,Apache-Spark/3.5.3 Delta-Lake/3.3.2


In [2]:
%load_ext sql

In [3]:
%%sql spark

select * from people00m;

RuntimeError: [TABLE_OR_VIEW_NOT_FOUND] The table or view `people00m` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 1 pos 14;
'Project [*]
+- 'UnresolvedRelation [people00m], [], false



In [4]:
%%sql

USE default;

Running query in 'SparkSession'

++
||
++
++

In [21]:
%%sql 

show tables;

Running query in 'SparkSession'

1 rows affected.

Field 1,Field 2,Field 3
default,people12m,False


In [7]:
%%sql 

DESCRIBE default.people12m

Running query in 'SparkSession'

RuntimeError: [TABLE_OR_VIEW_NOT_FOUND] The table or view `default`.`people12m` cannot be found. Verify the spelling and correctness of the schema and catalog.
If you did not qualify the name with a schema, verify the current_schema() output, or qualify the name with the correct schema and catalog.
To tolerate the error on drop use DROP VIEW IF EXISTS or DROP TABLE IF EXISTS.; line 1 pos 9;
'DescribeRelation false, [col_name#33, data_type#34, comment#35]
+- 'UnresolvedTableOrView [default, people12m], DESCRIBE TABLE, true



In [ ]:
%%sql 

DESCRIBE EXTENDED default.people12m

Running query in 'SparkSession'

15 rows affected.

Field 1,Field 2,Field 3
id,int,None
firstName,string,None


In [9]:
%%sql

CREATE OR REPLACE TABLE delta.`s3a://defaultfs/deltalake/people11m` (
  id INT,
  firstName STRING,
  middleName STRING,
  lastName STRING,
  gender STRING,
  birthDate TIMESTAMP,
  ssn STRING,
  salary INT
) USING DELTA


Running query in 'SparkSession'

++
||
++
++

In [12]:
%%sql 

CREATE TABLE default.people12m (
    id INT,
    firstName STRING,
    middleName STRING,
    lastName STRING,
    gender STRING,
    birthDate TIMESTAMP,
    ssn STRING,
    salary INT
)
USING DELTA
LOCATION 's3a://defaultfs/deltalake/people12m'

Running query in 'SparkSession'

++
||
++
++

In [ ]:
%%sql

CREATE DATABASE IF NOT EXISTS nyc;

Running query in 'SparkSession'

RuntimeError: If using snippets, you may pass the --with argument explicitly.
For more details please refer: https://jupysql.ploomber.io/en/latest/compose.html#with-argument


Original error message from DB driver:

[PARSE_SYNTAX_ERROR] Syntax error at or near 'CATALOG'.(line 1, pos 7)

== SQL ==
CREATE CATALOG IF NOT EXISTS nyc;
-------^^^




In [18]:
%%sql

USE nyc;

Running query in 'SparkSession'

++
||
++
++

In [19]:
%%sql 

show tables;

Running query in 'SparkSession'

++
||
++
++

#### Upsert to a Deltatable

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
from datetime import date

schema = StructType([
  StructField("id", IntegerType(), True),
  StructField("firstName", StringType(), True),
  StructField("middleName", StringType(), True),
  StructField("lastName", StringType(), True),
  StructField("gender", StringType(), True),
  StructField("birthDate", DateType(), True),
  StructField("ssn", StringType(), True),
  StructField("salary", IntegerType(), True)
])

data = [
  (9999998, 'Billy', 'Tommie', 'Luppitt', 'M', date.fromisoformat('1992-09-17'), '953-38-9452', 55250),
  (9999999, 'Elias', 'Cyril', 'Leadbetter', 'M', date.fromisoformat('1984-05-22'), '906-51-2137', 48500),
  (10000000, 'Joshua', 'Chas', 'Broggio', 'M', date.fromisoformat('1968-07-22'), '988-61-6247', 90000),
  (20000001, 'John', '', 'Doe', 'M', date.fromisoformat('1978-01-14'), '345-67-8901', 55500),
  (20000002, 'Mary', '', 'Smith', 'F', date.fromisoformat('1982-10-29'), '456-78-9012', 98250),
  (20000003, 'Jane', '', 'Doe', 'F', date.fromisoformat('1981-06-25'), '567-89-0123', 89900)
]

people_10m_updates = spark.createDataFrame(data, schema)
people_10m_updates.createOrReplaceTempView("people_10m_updates")

# ...

from delta.tables import DeltaTable

deltaTable = DeltaTable.forPath(spark, '/deltalake/peoples')

(deltaTable.alias("people_10m")
  .merge(
    people_10m_updates.alias("people_10m_updates"),
    "people_10m.id = people_10m_updates.id"
  )
  .whenMatchedUpdateAll()
  .whenNotMatchedInsertAll()
  .execute()
)

#### Read a Deltatable

In [ ]:
df = spark.read.format('delta').load("/deltalake/peoples")
df_filtered = df.filter(df["id"] >= 9999998)
display(df_filtered)

#### Write to a Deltatable

In [ ]:
# df.write.mode("append").saveAsTable("main.default.people_10m")

# Save as delta table
df.write.format('delta').mode('append').save('/deltalake/delta-table')

In [ ]:
# df.write.mode("overwrite").saveAsTable("main.default.people_10m")

# Save as delta table
df.write.format('delta').mode('overwrite').save('/deltalake/delta-table')

#### Update Deltatable Rows

In [ ]:
from delta import *
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

deltaTable = DeltaTable.forPath(spark, '/deltalake/peoples')

# Declare the predicate by using a SQL-formatted string.
deltaTable.update(
  condition = "gender = 'F'",
  set = { "gender": "'Female'" }
)

# Declare the predicate by using Spark SQL functions.
deltaTable.update(
  condition = col('gender') == 'M',
  set = { 'gender': lit('Male') }
)

#### Delete Rows 

In [ ]:
from delta.tables import *
from pyspark.sql.functions import *

deltaTable = DeltaTable.forPath(spark, '/deltalake/peoples')

# Declare the predicate by using a SQL-formatted string.
deltaTable.delete("birthDate < '1955-01-01'")

# Declare the predicate by using Spark SQL functions.
deltaTable.delete(col('birthDate') < '1960-01-01')

#### Display table history

In [ ]:
from delta.tables import *

deltaTable = DeltaTable.forPath(spark, '/deltalake/peoples')
display(deltaTable.history())

#### overwrite

In [ ]:
# Save as delta table
df.write.format('delta').mode('overwrite').save('/deltalake/delta-table')

#### Time Travle

In [ ]:
# Read version 1
from delta.tables import *

deltaTable = DeltaTable.forPath(spark, '/deltalake/peoples')
deltaHistory = deltaTable.history()

display(deltaHistory.where("version == 0"))
# Or:
display(deltaHistory.where("timestamp == '2024-05-15T22:43:15.000+00:00'"))

In [ ]:
df = spark.read.format('delta').option('versionAsOf', 0).load("/deltalake/peoples")
# Or: 2025-10-14 18:39:41
#df = spark.read.format('delta').option('timestampAsOf', '2025-10-14T18:45:03.000+00:00').load("/deltalake/peoples")

display(df)

#### Display table history

In [ ]:
deltaTable = DeltaTable.forPath(spark, "/deltalake/peoples")
print("######## Describe history for the table ######")
deltaTable.history().show()

#### Vacuum

In [ ]:
deltaTable = DeltaTable.forPath(spark, "/deltalake/peoples")
print("######## Vacuum the table ########")
deltaTable.vacuum()

#### Describe details for the table

In [ ]:
print("######## Describe details for the table ######")
deltaTable.detail().show()

#### Generating manifest 

In [ ]:
# Generate manifest
print("######## Generating manifest ######")
deltaTable.generate("SYMLINK_FORMAT_MANIFEST")

In [ ]:
# SQL Vacuum
print("####### SQL Vacuum #######")
spark.sql("VACUUM '%s' RETAIN 169 HOURS" % ("/deltalake/peoples")).collect()

In [ ]:
# SQL describe history
print("####### SQL Describe History ########")
print(spark.sql("DESCRIBE HISTORY delta.`%s`" % ("/deltalake/peoples")).collect())

In [ ]:
import shutil

# cleanup
shutil.rmtree("/tmp/delta-table")

In [ ]:
%%bash

aws --endpoint-url http://minio.sandbox.net:9010 s3 rm s3://defaultfs/deltalake/peoples --recursive

#### Optimize a table
After you have performed multiple changes to a table, you might have a lot of small files. To improve the speed of read queries, you can use the optimize operation to collapse small files into larger ones:

In [ ]:
from delta.tables import *

#deltaTable = DeltaTable.forName(spark, "main.default.people_10m")
deltaTable = DeltaTable.forPath(spark, "/deltalake/peoples")
deltaTable.optimize().executeCompaction()

#### Z-order by columns
To improve read performance further, you can collocate related information in the same set of files by z-ordering. Delta Lake data-skipping algorithms use this collocation to dramatically reduce the amount of data that needs to be read. To z-order data, you specify the columns to order on in the z-order by operation. For example, to collocate by gender, run:

In [ ]:
from delta.tables import *

#deltaTable = DeltaTable.forName(spark, "main.default.people_10m")
deltaTable = DeltaTable.forPath(spark, "/deltalake/peoples")
deltaTable.optimize().executeZOrderBy("gender")

#### Clean up snapshots with VACUUM
Delta Lake provides snapshot isolation for reads, which means that it is safe to run an optimize operation even while other users or jobs are querying the table. Eventually however, you should clean up old snapshots. You can do this by running the vacuum operation:

In [ ]:
from delta.tables import *

#deltaTable = DeltaTable.forName(spark, "main.default.people_10m")
deltaTable = DeltaTable.forPath(spark, "/deltalake/peoples")
deltaTable.vacuum()

#### How do I find the last commit's version in the Spark session?

In [ ]:
spark.conf.get("spark.databricks.delta.lastCommitVersionInSession")